# Cartesian Octree Read Example

Read 3d__var_4_n00005000.plt and build a Cartesian octree.


In [ ]:
from pathlib import Path
import sys
import tarfile


def repo_root() -> Path:
    """Find repository root by locating local package/data folders."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "example_data").is_dir() and (candidate / "batcamp").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root with example_data/ and batcamp/.")


REPO_ROOT = repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pooch
from starwinds_readplt.dataset import Dataset
from batcamp import Octree
from batcamp import format_octree_summary


_G2211_URL = "https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz"
_G2211_SHA256 = "c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136"
_ALLOWED_FILES = {'3d__var_4_n00000000.plt', '3d__var_4_n00044000.plt', '3d__var_4_n00005000.plt'}


def _unique_match(paths: list[Path], *, name: str) -> Path:
    if not paths:
        raise FileNotFoundError(name)
    if len(paths) > 1:
        raise FileNotFoundError(f"Expected unique match for {name}, found {len(paths)} entries: {paths}")
    return paths[0]


def _find_in_example_data(name: str) -> Path:
    return _unique_match(sorted((REPO_ROOT / "example_data").rglob(name)), name=name)


def _fetch_from_g2211_archive(name: str) -> Path:
    archive_path = Path(
        pooch.retrieve(
            url=_G2211_URL,
            known_hash=_G2211_SHA256,
            progressbar=False,
        )
    )
    with tarfile.open(archive_path, "r:gz") as tar:
        member_names = sorted(
            m.name for m in tar.getmembers() if m.isfile() and Path(m.name).name == name
        )
    member = _unique_match([Path(m) for m in member_names], name=name).as_posix()
    extracted = pooch.retrieve(
        url=_G2211_URL,
        known_hash=_G2211_SHA256,
        progressbar=False,
        processor=pooch.Untar(members=[member]),
    )
    if isinstance(extracted, (list, tuple)):
        extracted = extracted[0]
    return Path(extracted)


def data_file(name: str) -> Path:
    """Resolve one allowed sample file from local example_data or pooch."""
    if name not in _ALLOWED_FILES:
        raise ValueError(f"Unsupported sample file '{name}'. Allowed: {sorted(_ALLOWED_FILES)}")
    try:
        return _find_in_example_data(name)
    except FileNotFoundError:
        return _fetch_from_g2211_archive(name)



In [ ]:
data_path = data_file('3d__var_4_n00005000.plt')
print(data_path)
assert data_path.exists()


In [ ]:
ds = Dataset.from_file(str(data_path))
print(ds)


In [ ]:
tree = Octree.from_dataset(ds, tree_coord='xyz')
print(format_octree_summary(tree))
print('levels:', tree.levels)
print('is_uniform:', tree.is_uniform)


In [ ]:
center = tree.lookup._cell_centers[len(tree.lookup._cell_centers) // 2]
hit = tree.lookup_point(center, space='xyz')
print('center:', center)
print('cell_id:', None if hit is None else hit.cell_id)
print('level:', None if hit is None else hit.level)


## XY Plane Resample (log rho)

Resample on z=0 with the octree interpolator and plot log10 of rho.


In [ ]:
from batcamp import OctreeInterpolator
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from batcamp import OctreeInterpolator
import numpy as np
import matplotlib.pyplot as plt

rho_name = next(name for name in ds.variables if name.startswith('Rho '))
interp = OctreeInterpolator(ds, [rho_name], tree=tree)

centers = np.asarray(tree.lookup._cell_centers, dtype=float)
xmin, xmax = float(np.min(centers[:, 0])), float(np.max(centers[:, 0]))
ymin, ymax = float(np.min(centers[:, 1])), float(np.max(centers[:, 1]))

nx, ny = 300, 300
x = np.linspace(xmin, xmax, nx)
y = np.linspace(ymin, ymax, ny)
xx, yy = np.meshgrid(x, y, indexing='xy')
zz = np.zeros_like(xx)
q = np.column_stack((xx.ravel(), yy.ravel(), zz.ravel()))

rho, cell_ids = interp(q, query_coord='xyz', return_cell_ids=True)
rho = np.asarray(rho, dtype=float).reshape(ny, nx)
cell_ids = np.asarray(cell_ids, dtype=int).reshape(ny, nx)

mask = cell_ids < 0
rho = np.where(mask, np.nan, rho)
rho = np.where(rho > 0.0, rho, np.nan)
log_rho = np.log10(rho)

print('rho field:', rho_name)
print('grid:', (ny, nx))
print('valid fraction:', float(np.mean(~np.isnan(log_rho))))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
im = ax.pcolormesh(xx, yy, log_rho, shading='auto', cmap='viridis')
ax.set_xlabel('X [R]')
ax.set_ylabel('Y [R]')
ax.set_title(f'log10({rho_name}) on z=0')
ax.set_aspect('equal')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('log10 rho')
plt.show()


## Ray Demo: Column Quantity

Sample a very small set of rays through the Cartesian domain, compute `2 * rho * temp` along each ray, and integrate along path length to get a column value per ray.

If a temperature variable is missing in this file, use `P/rho` as a temperature proxy for the demo.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from batcamp import OctreeInterpolator
from batcamp import OctreeRayInterpolator

# Fast column demo: integrate 2*P along +X using batched ray integration.
p_name = next(name for name in ds.variables if name.startswith('P '))
p_interp = OctreeInterpolator(ds, [p_name], tree=tree)
ray_interp = OctreeRayInterpolator(p_interp)

points_xyz = np.asarray(tree.lookup._points, dtype=float)
dmin = points_xyz.min(axis=0)
dmax = points_xyz.max(axis=0)

xmin, ymin, zmin = dmin
xmax, ymax, zmax = dmax

yc = 0.5 * (ymin + ymax)
zc = 0.5 * (zmin + zmax)
y_span = 0.35 * (ymax - ymin)
z_span = 0.35 * (zmax - zmin)

ny, nz = 5, 5
y_vals = np.linspace(yc - y_span, yc + y_span, ny)
z_vals = np.linspace(zc - z_span, zc + z_span, nz)

ray_dir = np.array([1.0, 0.0, 0.0], dtype=float)
L = float(xmax - xmin)

origins = np.empty((ny * nz, 3), dtype=float)
k = 0
for z0 in z_vals:
    for y0 in y_vals:
        origins[k] = (xmin, y0, z0)
        k += 1

# scale=2.0 gives integral of 2*P along the ray.
col_flat = ray_interp.integrate_field_along_rays(
    origins,
    ray_dir,
    0.0,
    L,
    chunk_size=4096,
    scale=2.0,
)
col_img = np.asarray(col_flat, dtype=float).reshape(nz, ny)

fig, ax = plt.subplots(figsize=(6.8, 5.5), constrained_layout=True)
im = ax.imshow(
    col_img,
    origin='lower',
    extent=[y_vals.min(), y_vals.max(), z_vals.min(), z_vals.max()],
    aspect='auto',
    cmap='viridis',
)
ax.set_xlabel('Ray start Y [R]')
ax.set_ylabel('Ray start Z [R]')
ax.set_title(f'Column image (5x5 rays): integral of 2*{p_name} along +X')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Column integral')
plt.show()

print('computed quantity: integral of 2*P along ray (equivalent to 2*rho*(P/rho))')
print('pressure field:', p_name)
print('grid shape:', col_img.shape)
print('column min/max:', float(np.nanmin(col_img)), float(np.nanmax(col_img)))
